In [10]:
from pathlib import Path
import sys
import numpy as np
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor
from functools import partial

In [11]:
from pathlib import Path
import sys

ROOT = Path.cwd().parents[1]

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Project root:", ROOT)


Project root: /home/sangonvi/Cefet/repositories/rionowcastdf


In [12]:
from src.corrdiff.builder.CorrdiffDatasetBuilder import RadarDataset
from src.config.paths import (
    ERA5_DIR,
    RADAR_CACHE_DIR,
    RADAR_DIR,
)

In [13]:
def process_timestamp(
    timestamp,
    radar,
):

    grid = radar.process_time(timestamp)

    if grid is None:
        return None

    valid = np.isfinite(grid)

    if not valid.any():
        return None

    rain_mask = grid > 0

    values = grid[rain_mask]

    H, W = grid.shape

    sum_grid = np.zeros(
        (H, W),
        dtype=np.float32,
    )

    count_grid = np.zeros(
        (H, W),
        dtype=np.uint32,
    )

    occurrence = np.zeros(
        (H, W),
        dtype=np.uint32,
    )

    sum_grid[rain_mask] = values

    count_grid[rain_mask] = 1

    occurrence[rain_mask] = 1

    if len(values) == 0:

        row = {
            "timestamp": timestamp,
            "rain_fraction": 0.0,
            "rain_pixels": 0,
            "mean": 0.0,
            "std": 0.0,
            "max": 0.0,
            "volume": 0.0,
        }

    else:

        row = {
            "timestamp": timestamp,
            "rain_fraction":
                len(values) / grid.size,

            "rain_pixels":
                int(len(values)),

            "mean":
                float(values.mean()),

            "std":
                float(values.std()),

            "max":
                float(values.max()),

            "volume":
                float(values.sum()),
        }

    return {
        "row": row,
        "sum_grid": sum_grid,
        "count_grid": count_grid,
        "occurrence": occurrence,
        "max_grid": grid.astype(np.float32),
    }

In [15]:
radar = RadarDataset(
    radar_path=RADAR_DIR,
    cache_dir=RADAR_CACHE_DIR,
    resolution_km=2,
    lat_range=[-23.5, -22.25],
    lon_range=[-44.0, -42.5],
    start_date="2011-01-01",
    end_date="2024-12-31",
)

radar.build_grid()
radar.precompute_pixel_map()

timestamps = radar.available_timestamps()

print(
    f"Found {len(timestamps):,} timestamps"
)

[2026-06-22 01:25:55,668] [INFO] Grid shape: (70, 84)
[2026-06-22 01:25:55,680] [INFO] PX range: 0 - 653
[2026-06-22 01:25:55,681] [INFO] PY range: 3 - 653
[2026-06-22 01:27:10,630] [INFO] Available radar timestamps: 2836176
Found 2,836,176 timestamps


In [ ]:
MAX_WORKERS = 16

worker = partial(
    process_timestamp,
    radar=radar,
)

with ProcessPoolExecutor(
    max_workers=MAX_WORKERS
) as executor:

    iterator = executor.map(
        worker,
        timestamps,
        chunksize=100,
    )

    for result in tqdm(
        iterator,
        total=len(timestamps),
    ):

        if result is None:
            continue

        row = result["row"]

        buffer.append(row)

        sum_grid += result["sum_grid"]

        count_grid += result["count_grid"]

        rain_occurrence += result["occurrence"]

        max_grid = np.maximum(
            max_grid,
            result["max_grid"]
        )

        processed_images += 1

        if len(buffer) >= BUFFER_SIZE:

            df = pd.DataFrame(buffer)

            table = pa.Table.from_pandas(
                df,
                preserve_index=False,
            )

            if writer is None:

                writer = pq.ParquetWriter(
                    str(PARQUET_FILE),
                    table.schema,
                    compression="snappy",
                )

            writer.write_table(table)

            buffer.clear()

01 - Radar Global Statistics

In [ ]:

radar_res = 2
lat_range = [-23.5, -22.25]
lon_range = [-44.0, -42.5]
begin = "2011-01"
end =  "2024-12"
radar = RadarDataset(
        radar_path=RADAR_DIR,
        cache_dir=RADAR_CACHE_DIR,
        resolution_km=radar_res,
        lat_range=lat_range,
        lon_range=lon_range,
        start_date="2011-01-01",
        end_date="2024-12-31",
    )

radar.build_grid()
radar.precompute_pixel_map()
times = radar.available_timestamps()

sample = []

for timestamp in tqdm(times):
    grid = radar.process_time(timestamp)

    if grid is None:
        continue

    values = grid.flatten()

    values = values[values > 0]

    if len(values) == 0:
        continue

    if len(values) > 1000:
        idx = np.random.choice(
            len(values),
            1000,
            replace=False
        )

        values = values[idx]

    sample.append(values)
sample = np.concatenate(sample)

print(np.percentile(sample,50))
print(np.percentile(sample,90))
print(np.percentile(sample,99))
print(np.percentile(sample,99.9))

[2026-06-21 22:35:24,378] [INFO] Grid shape: (70, 84)
[2026-06-21 22:35:24,382] [INFO] PX range: 0 - 653
[2026-06-21 22:35:24,383] [INFO] PY range: 3 - 653
[2026-06-21 22:36:37,511] [INFO] Available radar timestamps: 2836176


  0%|          | 1122/2836176 [00:00<40:58, 1153.08it/s]

[2026-06-21 22:36:38,667] [ERROR] Error reading image /home/sangonvi/Cefet/repositories/rionowcastdf/data/radar_sumare/2011/01/21/2011_01_21_08_40.png: cannot identify image file '/home/sangonvi/Cefet/repositories/rionowcastdf/data/radar_sumare/2011/01/21/2011_01_21_08_40.png'
[2026-06-21 22:36:38,674] [ERROR] Error reading image /home/sangonvi/Cefet/repositories/rionowcastdf/data/radar_sumare/2011/01/21/2011_01_21_08_54.png: cannot identify image file '/home/sangonvi/Cefet/repositories/rionowcastdf/data/radar_sumare/2011/01/21/2011_01_21_08_54.png'


  0%|          | 3189/2836176 [06:50<101:13:08,  7.77it/s]


KeyboardInterrupt: 